# GPT 퀘스트 - Decoder-only Transformer 구현

**날짜:** 2026년 3월 6일  
**목표:** 기존 Encoder-Decoder Transformer를 GPT(Decoder-only) 구조로 변환하여 프리트레인 수행

---
## 1번 문제: Transformer와 GPT의 아키텍처 차이점

### 변경 사항 요약

| 항목 | 기존 Transformer (Encoder-Decoder) | GPT (Decoder-only) |
|------|-----------------------------------|-----------------------|
| 구조 | Encoder + Decoder | Decoder만 사용 |
| Attention | Self-Attn + Cross-Attn (Encoder-Decoder Attn) | Masked Self-Attn만 사용 |
| 위치 정보 | Sinusoidal Positional Encoding (고정) | Positional Embedding (학습 가능, `nn.Embedding`) |
| 학습 방식 | Q→A 정답 쌍 필요 | 단일 문장, Next Token Prediction |
| 데이터 형식 | `(질문, 답변)` 쌍 | `<start> Q <sep> A <end>` 단일 시퀀스 |

### 코드 변경 핵심

**① Encoder 제거 — Decoder-only 구조**
```python
# 기존: Encoder + Decoder
self.transformer_encoder = nn.TransformerEncoder(...)
self.transformer_decoder = nn.TransformerDecoder(...)

# 변경: TransformerEncoder를 Decoder-only로 사용 (causal mask로 autoregressive 보장)
self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
```

**② Sinusoidal → Learnable Positional Embedding**
```python
# 기존: Sine/Cosine 함수 기반 고정 인코딩
self.positional_encoding = self._get_positional_encoding(d_model, max_len)

# 변경: 학습 가능한 임베딩
self.position_embedding = nn.Embedding(max_seq_length, d_model)
```

**③ Dataset 변경 — Next Token Prediction**
```python
# 기존: Q/A 쌍
return {'question': q_tensor, 'answer': a_tensor}

# 변경: 단일 시퀀스, input/target shift
sequence = [<start>] + Q_tokens + [<sep>] + A_tokens + [<end>]
input_ids  = sequence[:-1]
target_ids = sequence[1:]   # next token prediction
```

---
## 2번 문제: 데이터 전처리

- Decoder 기반 생성모델에 맞게 챗봇 데이터를 변환
- `<start> Q <sep> A <end>` 형식으로 delimiter를 붙여 단일 시퀀스 구성
- 이번 과제는 pretrain을 위한 데이터이므로 정답 쌍 불필요

In [23]:
import os
os.chdir('/workspace/test/NLP/NLP03/chatgptbot')
print(os.getcwd())

/workspace/test/NLP/NLP03/chatgptbot


In [24]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import pickle

# 데이터 로드
df = pd.read_csv('./data/processed/augmented_data.csv')
print(f'데이터 크기: {len(df)}')
print(df.head())

데이터 크기: 35247
          question       answer  label
0           12시 땡!   하루가 또 가네요.    0.0
1      1지망 학교 떨어졌어    위로해 드립니다.    0.0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.    0.0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.    0.0
4          PPL 심하네   눈살이 찌푸려지죠.    0.0


In [25]:
# 코퍼스 및 토크나이저 로드
with open('./data/processed/corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

with open('./models/tokenizer.pkl', 'rb') as f:
    tok = pickle.load(f)

word2idx = tok['word2idx'] if isinstance(tok, dict) else tok.word2idx
print(f'어휘 크기: {len(word2idx)}')
print(f'특수 토큰 확인: <start>={word2idx.get("<start>")}, <sep>={word2idx.get("<sep>")}, <end>={word2idx.get("<end>")}')

어휘 크기: 10005
특수 토큰 확인: <start>=2, <sep>=10004, <end>=3


In [26]:
# GPTDataset 시퀀스 구성 예시
sample_q = '오늘 기분이 어때'
sample_a = '좋아요 감사합니다'

sequence = ['<start>'] + sample_q.split() + ['<sep>'] + sample_a.split() + ['<end>']
print('시퀀스:', sequence)
print('input_ids:', sequence[:-1])
print('target_ids:', sequence[1:])

시퀀스: ['<start>', '오늘', '기분이', '어때', '<sep>', '좋아요', '감사합니다', '<end>']
input_ids: ['<start>', '오늘', '기분이', '어때', '<sep>', '좋아요', '감사합니다']
target_ids: ['오늘', '기분이', '어때', '<sep>', '좋아요', '감사합니다', '<end>']


---
## 3번 문제: 위치 정보 추가 (Positional Embedding)

- 기존 Sinusoidal Encoding → 학습 가능한 `nn.Embedding`으로 변경
- 데이터의 위치 정보를 토큰 임베딩에 더하여 순서 정보 제공

In [27]:
import torch
import torch.nn as nn
import importlib.util

spec = importlib.util.spec_from_file_location('train', './scripts/04_train.py')
tm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tm)

import config

vocab_size = len(word2idx)
model = tm.GPTModel(
    vocab_size=vocab_size,
    d_model=config.TRANSFORMER_D_MODEL,
    nhead=config.TRANSFORMER_NHEAD,
    num_layers=config.TRANSFORMER_NUM_LAYERS,
    dim_feedforward=config.TRANSFORMER_DIM_FEEDFORWARD,
    dropout=0.0,
    max_seq_length=config.MAX_SEQ_LENGTH
)

# Positional Embedding 확인
print('Position Embedding 타입:', type(model.position_embedding))
print('Position Embedding 크기:', model.position_embedding.weight.shape)
print('→ (max_seq_length=50, d_model=768) — 학습 가능한 파라미터')

Using device: cuda
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 25.26GB


Position Embedding 타입: <class 'torch.nn.modules.sparse.Embedding'>
Position Embedding 크기: torch.Size([50, 768])
→ (max_seq_length=50, d_model=768) — 학습 가능한 파라미터


---
## 4번 문제: 모델 구현 확인 (print(model))

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, map_location=device))
model.to(device)
model.eval()

print(model)
print(f'\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

GPTModel(
  (token_embedding): Embedding(10005, 768, padding_idx=0)
  (position_embedding): Embedding(50, 768)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (fc_out): Linear(in_features=768, out_features=10005, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

총 파라미터 수: 57,943,317


/tmp/ipykernel_19221/3284911154.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, map_location=device))


---
## 5번 문제: 모델 훈련 결과 및 텍스트 생성

### 훈련 결과 (실험 #8~#10)

In [33]:
import json

with open('./models/training_results.json', 'r') as f:
    results = json.load(f)

print('=== 훈련 결과 요약 ===')
print(f'총 에폭: {results["epochs"]}')
print(f'Best Val Loss: {results["best_val_loss"]:.4f}')
print(f'Final Train Loss: {results["final_train_loss"]:.4f}')
print(f'Final Val Loss: {results["final_val_loss"]:.4f}')

=== 훈련 결과 요약 ===
총 에폭: 20
Best Val Loss: 3.3220


KeyError: 'final_train_loss'

In [30]:
# 에폭별 손실 출력
print('에폭별 Train Loss / Val Loss')
print('-' * 40)
for i, (tr, vl) in enumerate(zip(results['train_losses'], results['val_losses']), 1):
    print(f'Epoch {i:2d}: Train={tr:.4f}  Val={vl:.4f}')

에폭별 Train Loss / Val Loss
----------------------------------------
Epoch  1: Train=5.3018  Val=4.7373
Epoch  2: Train=4.5855  Val=4.5083
Epoch  3: Train=4.3280  Val=4.2410
Epoch  4: Train=4.0593  Val=4.0852
Epoch  5: Train=3.8297  Val=3.9262
Epoch  6: Train=3.6195  Val=3.8195
Epoch  7: Train=3.4100  Val=3.6977
Epoch  8: Train=3.2240  Val=3.6147
Epoch  9: Train=3.0339  Val=3.5664
Epoch 10: Train=2.8776  Val=3.5118
Epoch 11: Train=2.7263  Val=3.4530
Epoch 12: Train=2.5820  Val=3.4189
Epoch 13: Train=2.4461  Val=3.4001
Epoch 14: Train=2.3404  Val=3.3701
Epoch 15: Train=2.2346  Val=3.3452
Epoch 16: Train=2.1481  Val=3.3387
Epoch 17: Train=2.0662  Val=3.3355
Epoch 18: Train=1.9996  Val=3.3276
Epoch 19: Train=1.9477  Val=3.3241
Epoch 20: Train=1.9090  Val=3.3220


In [31]:
from konlpy.tag import Okt
_okt = Okt()
def okt_tokenize(text):
    return _okt.morphs(text)

In [32]:
import torch, pickle, sys, os, importlib.util
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import config
spec = importlib.util.spec_from_file_location("train", os.path.join(os.path.dirname(__file__), "04_train.py"))
tm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tm)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(config.TOKENIZER_PATH, "rb") as f:
    tok = pickle.load(f)
word2idx = tok["word2idx"] if isinstance(tok, dict) else tok.word2idx
idx2word = tok["idx2word"] if isinstance(tok, dict) else tok.idx2word
vocab_size = len(word2idx)
model = tm.GPTModel(vocab_size=vocab_size, d_model=config.TRANSFORMER_D_MODEL, nhead=config.TRANSFORMER_NHEAD, num_layers=config.TRANSFORMER_NUM_LAYERS, dim_feedforward=config.TRANSFORMER_DIM_FEEDFORWARD, dropout=0.0, max_seq_length=config.MAX_SEQ_LENGTH)
model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, map_location=device))
model.to(device)
model.eval()
print(model)
def generate(prompt, max_new_tokens=30):
    ids = [word2idx.get(t, 1) for t in prompt.split()]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            mask = torch.triu(torch.ones(x.size(1), x.size(1)), diagonal=1).bool().to(device)
            logits = model(x, None, mask)
            nt = logits[0, -1, :].argmax().item()
            if idx2word.get(nt) in ["<end>", "<pad>"]: break
            x = torch.cat([x, torch.tensor([[nt]]).to(device)], dim=1)
    return " ".join([idx2word.get(i, "<unk>") for i in x[0].tolist()])
for p in ["오늘 기분이", "밥 먹었어", "너무 힘들어"]:
    print(f"입력: {p}\n출력: {generate(p)}\n")

NameError: name '__file__' is not defined